# Chuncking Documents

## Splitting into chunks

In [1]:
from ai_eng.settings import KNOWLEDGE_DIR
from ai_eng.loaders import load_corpus

docs = load_corpus(KNOWLEDGE_DIR)

d:\coding shuttle\ai engineer\practice-lab\ai-eng\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.


Loaded 78 documents from D:\coding shuttle\ai engineer\practice-lab\ai-eng\src\ai_eng\4_langchain_rag\knowledge
  skipped vendor_contract.pdf: tesseract is not installed or it's not in your PATH. See README file for more information.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,         # maximum characters per chunk
    chunk_overlap=120,      # characters shared between neighbouring chunks
    add_start_index=True,   # remembers where in the page each chunk began
)

chunks = splitter.split_documents(docs)

print(f"Split {len(docs)} documents into {len(chunks)} chunks")
print(chunks[0].page_content[:200])
print(chunks[0].metadata)

Split 78 documents into 102 chunks
question: How do I get a refund for a cancelled order?
answer: Refunds for cancelled orders go back to the original payment method within 5 to 7 working days. Zomato credits are refunded instantly.
{'source': 'D:\\coding shuttle\\ai engineer\\practice-lab\\ai-eng\\src\\ai_eng\\4_langchain_rag\\knowledge\\faq\\support_faq.csv', 'row': 0, 'product': 'Zomato App', 'category': 'Refunds', 'filename': 'support_faq.csv', 'area': 'faq', 'doc_type': 'csv', 'start_index': 0}


In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

text = open("knowledge/wiki/onboarding.md", encoding="utf-8").read()

# 1. cuts only at blank lines, then merges paragraphs up to the size limit
by_char = CharacterTextSplitter(separator="\n\n", chunk_size=500, chunk_overlap=50)

# 2. tries blank lines, then newlines, then spaces, so oversized paragraphs still get split
by_recursive = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# 3. sizes in tokens, matching what the embedding model actually counts
by_tokens = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=120, chunk_overlap=15
)

for name, splitter in [("character", by_char), ("recursive", by_recursive), ("tokens", by_tokens)]:
    pieces = splitter.split_text(text)
    longest = max(len(p) for p in pieces)
    print(f"{name:>10}: {len(pieces):>3} chunks, longest = {longest} characters")

 character:  15 chunks, longest = 479 characters
 recursive:  15 chunks, longest = 479 characters
    tokens:  12 chunks, longest = 596 characters


In [1]:
%pip install tiktoken
import tiktoken            # pip install tiktoken

enc = tiktoken.get_encoding("cl100k_base")      # the encoding used by text-embedding-3-small
ids = enc.encode("Zomato delivers unbelievably fast.")

print(ids)                                  # [57, 316, 4428, 28421, 40037, 89234, 5043, 13]
print([enc.decode([i]) for i in ids])       # ['Z', 'om', 'ato', ' delivers', ' unbelie', 'vably', ' fast', '.']
print(len(ids), "tokens")                   # 8 tokens

Note: you may need to restart the kernel to use updated packages.
[57, 316, 4428, 28421, 40037, 89234, 5043, 13]
['Z', 'om', 'ato', ' delivers', ' unbelie', 'vably', ' fast', '.']
8 tokens


## Header-aware splitting for Markdown and HTML

In [13]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
)

markdown = open("knowledge/wiki/onboarding.md", encoding="utf-8").read()
sections = header_splitter.split_text(markdown)

print(len(sections), "sections")
print(sections[2].metadata)
print(sections[2].page_content[:150])

17 sections
{'h1': 'Engineering Onboarding', 'h2': 'Accounts and access'}
You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access 


In [12]:
size_cap = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120, add_start_index=True)
chunks = size_cap.split_documents(sections)      # sections that already fit stay as they are
for chunk in chunks:
    print(f"{chunk}\n")

page_content='Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.' metadata={'h1': 'Engineering Onboarding', 'start_index': 0}

page_content='- Collect your laptop and access card from the IT desk.
- Meet your onboarding buddy. They will walk you through the team's codebase and rituals.
- Complete the mandatory security awareness course in the Learning Portal within your first 3 days.
- Join the team Slack channels listed in your welcome email.' metadata={'h1': 'Engineering Onboarding', 'h2': 'Day one', 'start_index': 0}

page_content='You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access is **not** granted automatically. Request it through the Access Portal after you finish the security course; your manager must approve it, and access is reviewed every quarter.' metadata={'h1': 'Engineering Onboarding', 'h2': 'Accounts and access', '

In [14]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=[("h1", "title"), ("h2", "section")])
parts = html_splitter.split_text(open("knowledge/wiki/holiday_calendar.html", encoding="utf-8").read())

for part in parts:
    print(f"{part}\n")

page_content='Company holidays' metadata={'title': 'Company holidays'}

page_content='This calendar applies to all corporate offices. Customer Support and Delivery Operations follow a reduced rota on holidays, and employees who work on a holiday receive a compensatory off.' metadata={'title': 'Company holidays'}

page_content='Fixed-date holidays' metadata={'title': 'Company holidays', 'section': 'Fixed-date holidays'}

page_content='Holiday  
Date  
Republic Day  
26 January  
Independence Day  
15 August  
Gandhi Jayanti  
2 October  
Christmas Day  
25 December' metadata={'title': 'Company holidays', 'section': 'Fixed-date holidays'}

page_content='Festival holidays' metadata={'title': 'Company holidays', 'section': 'Festival holidays'}

page_content='Holi, Eid and Diwali dates follow the lunar calendar. People Operations publishes the exact dates every December. Offices stay closed on these days. Diwali is a two-day holiday.' metadata={'title': 'Company holidays', 'section': 'Festi

## Splitting code and JSON

In [23]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

source = open("knowledge/wiki/retry_helper.py", encoding="utf-8").read()

code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=600,
    chunk_overlap=60,
)
code_chunks = code_splitter.create_documents([source])
print(len(code_chunks), "chunks, each ending near a function boundary")

6 chunks, each ending near a function boundary


In [29]:
import json
from langchain_text_splitters import RecursiveJsonSplitter

data = json.load(open("knowledge/catalog/plans.json", encoding="utf-8"))
print(data)

json_splitter = RecursiveJsonSplitter(max_chunk_size=400)
json_docs = json_splitter.create_documents(texts=[data])
print(len(json_docs), "JSON chunks")

for chunk in json_docs:
    print(f"{chunk}\n")

{'product': 'Zomato for Enterprise', 'currency': 'INR', 'billing': {'cycle': 'monthly', 'invoice': 'one consolidated GST invoice', 'payment_terms_days': 30, 'annual_commitment_discount_percent': 10}, 'plans': [{'name': 'Starter', 'price_per_employee_per_month': 99, 'min_employees': 10, 'sso': False, 'scim': False, 'support': 'Email support, response within 2 business days', 'features': ['Company meal wallet', 'Daily spending limits', 'Monthly consolidated invoice', 'Email support']}, {'name': 'Business', 'price_per_employee_per_month': 199, 'min_employees': 50, 'sso': True, 'scim': False, 'support': 'Chat support, 9am to 9pm, response within 4 hours', 'features': ['Everything in Starter', 'Single sign-on (SAML 2.0)', 'Department-wise reports', 'Allowed time windows and restaurant rules', 'Scheduled usage reports by email', 'Chat support, 9am to 9pm']}, {'name': 'Enterprise', 'price_per_employee_per_month': 349, 'min_employees': 500, 'sso': True, 'scim': True, 'support': '24x7 priority 

## Semantic chunking

In [31]:
%pip install langchain-experimental

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

semantic = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",     # cut at the biggest drops in similarity
)

chunks = semantic.split_documents(docs[:5])
print(len(chunks), "semantic chunks")

## One chunking module for every document type


In [33]:
"""Turn Documents into chunks, using the splitter that suits each document type."""
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

_by_size = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120, add_start_index=True)
_by_header = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")])


def _split_markdown(doc: Document) -> list[Document]:
    sections = _by_header.split_text(doc.page_content)
    for section in sections:
        section.metadata = {**doc.metadata, **section.metadata}   # keep file info, add headings
    return _by_size.split_documents(sections)


def chunk_corpus(docs: list[Document]) -> list[Document]:
    chunks: list[Document] = []
    for doc in docs:
        kind = doc.metadata["doc_type"]
        if kind == "md":
            chunks.extend(_split_markdown(doc))
        elif kind == "csv":
            chunks.append(doc)                       # a row is already one answer
        else:
            chunks.extend(_by_size.split_documents([doc]))
    print(f"{len(docs)} documents became {len(chunks)} chunks")
    return chunks

## Metadata and stable chunk IDs


In [38]:
import hashlib
import uuid
from langchain_core.documents import Document

def generate_stable_id(chunk: Document) -> str:
    """
    Generates a deterministic ID based on the file name and chunk content.
    """
    # 1. Get source filename (or fallback to 'unknown')
    filename = chunk.metadata.get("filename", "unknown")
    
    # 2. Hash the chunk's text content
    text_hash = hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest()
    
    # 3. Combine filename and text hash into a deterministic UUID (v5)
    unique_string = f"{filename}|{text_hash}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, unique_string))


# --- Example Usage ---

# Create dummy chunks
chunk1 = Document(
    page_content="Full-time employees receive 24 days of paid annual leave.",
    metadata={"filename": "employee_handbook.pdf", "page": 1}
)

chunk2 = Document(
    page_content="Up to 5 unused days may be carried over into the next year.",
    metadata={"filename": "employee_handbook.pdf", "page": 1}
)

# Generate IDs
id1_run1 = generate_stable_id(chunk1)
id2_run1 = generate_stable_id(chunk2)

print("First Run IDs:")
print(f"Chunk 1 ID: {id1_run1}")
print(f"Chunk 2 ID: {id2_run1}")

# Simulate re-running ingestion on Chunk 1
id1_run2 = generate_stable_id(chunk1)

print("\nSecond Run ID (Re-ingesting Chunk 1):")
print(f"Chunk 1 ID: {id1_run2}")

# Verify stability
print(f"\nIs the ID identical across runs? {id1_run1 == id1_run2}")

First Run IDs:
Chunk 1 ID: c67646ad-2e2c-5e60-b74b-56d731672fa6
Chunk 2 ID: 8ed017e3-a930-5936-a3f2-cad3f970e6a2

Second Run ID (Re-ingesting Chunk 1):
Chunk 1 ID: c67646ad-2e2c-5e60-b74b-56d731672fa6

Is the ID identical across runs? True
